# AudioEditor - Cloud Stem Separation mit htdemucs GPU

Dieses Notebook implementiert den automatisierten Workflow für den Audio Editor.

- **Hardware**: GPU zwingend (T4, V100, A100)
- **Modell**: Hybrid Transformer Demucs (htdemucs)
- **Gewichte**: Offiziell via demucs Bibliothek (keine manuellen Links)
- **Input**: /MyDrive/AudioEditor_Stems/Input/ (Uploads vom Desktop Editor)
- **Output**: /MyDrive/AudioEditor_Stems/Output/<Track_ID>/ (drums, bass, other, vocals)

Der Watcher beobachtet den Input Ordner und verarbeitet neue Tracks automatisch.


## 1. Google Drive mounten

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive gemountet: /content/drive/MyDrive")


## 2. GPU prüfen (zwingend laut Spec)

In [ ]:
import torch
import json

print("=== GPU Check - Muss GPU sein laut Spec ===")
cuda_available = torch.cuda.is_available()
print(f"CUDA verfügbar: {cuda_available}")
print(f"PyTorch Version: {torch.__version__}")

if cuda_available:
    print(f"Device Count: {torch.cuda.device_count()}")
    print(f"Aktuelles Device: {torch.cuda.current_device()}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Total Memory: {props.total_memory / 1024**3:.2f} GB")
    print("✅ GPU verfügbar - bereit für htdemucs")
else:
    print("⚠️ Keine GPU! Bitte aktivieren:")
    print("Laufzeit -> Laufzeittyp ändern -> Hardwarebeschleuniger: GPU")
    raise RuntimeError("GPU erforderlich für htdemucs, aber nicht verfügbar")


## 3. Dependencies installieren
Demucs und Torch mit offiziellen Gewichten

In [ ]:
!pip install -q demucs torch torchaudio
print("Dependencies installiert")


## 4. Ordnerstruktur prüfen/erstellen

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/AudioEditor_Stems")
INPUT_DIR = BASE / "Input"
OUTPUT_DIR = BASE / "Output"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base: {BASE}")
print(f"Input: {INPUT_DIR} - Exists: {INPUT_DIR.exists()}")
print(f"Output: {OUTPUT_DIR} - Exists: {OUTPUT_DIR.exists()}")
print(f"\nInput Dateien: {list(INPUT_DIR.glob('*'))[:10]}")


## 5. Demucs Processor - htdemucs Modell mit GPU
Lädt Gewichte offiziell über demucs Bibliothek, keine manuellen Links

In [ ]:
import torch
import torchaudio
from pathlib import Path
import json
import shutil
from datetime import datetime
import time

STANDARD_STEMS = ["drums", "bass", "other", "vocals"]

def check_gpu():
    cuda_available = torch.cuda.is_available()
    info = {
        "cuda_available": cuda_available,
        "device_count": torch.cuda.device_count() if cuda_available else 0,
    }
    if cuda_available:
        info["device_name"] = torch.cuda.get_device_name(0)
    print(json.dumps(info, indent=2))
    return info

class DemucsProcessor:
    def __init__(self, output_base: Path):
        self.output_base = Path(output_base)
        self.output_base.mkdir(parents=True, exist_ok=True)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"[Demucs] Device: {self.device}")
        
        # Modell laden - offizielle Gewichte via demucs Bibliothek
        print("[Demucs] Lade htdemucs Modell (offizielle Gewichte)...")
        from demucs.pretrained import get_model
        self.model = get_model("htdemucs")
        self.model.to(self.device)
        print(f"[Demucs] Modell bereit: htdemucs auf {self.device}")
        print(f"[Demucs] Sources: {self.model.sources}")
        print(f"[Demucs] Samplerate: {self.model.samplerate}")

    def separate_track(self, input_path: Path, track_id: str):
        input_path = Path(input_path)
        output_dir = self.output_base / track_id
        output_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"\n[Demucs] Starte Separation: {input_path.name} (ID: {track_id})")
        print(f"[Demucs] Output: {output_dir}")
        
        status_path = output_dir / "status.json"
        status = {
            "track_id": track_id,
            "input_file": str(input_path),
            "status": "processing",
            "model": "htdemucs",
            "device": self.device,
            "started_at": datetime.now().isoformat(),
            "stems": STANDARD_STEMS
        }
        with open(status_path, 'w') as f:
            json.dump(status, f, indent=2)
        
        try:
            # Audio laden
            from demucs.audio import AudioFile
            from demucs.apply import apply_model
            
            print(f"[Demucs] Lade Audio: {input_path}")
            wav = AudioFile(str(input_path)).read(
                streams=0,
                samplerate=self.model.samplerate,
                channels=self.model.audio_channels
            )
            wav = wav.to(self.device)
            print(f"[Demucs] Audio shape: {wav.shape}")
            
            # Normalisieren
            ref = wav.mean(0)
            wav = (wav - ref.mean()) / ref.std()
            
            # Separation mit GPU
            print(f"[Demucs] Starte Separation auf {self.device}...")
            with torch.no_grad():
                sources = apply_model(self.model, wav[None], device=self.device, split=True, overlap=0.25, progress=True)[0]
            
            print(f"[Demucs] Separation done: {sources.shape}")
            
            # Denormalisieren und speichern
            sources = sources * ref.std() + ref.mean()
            
            for i, stem_name in enumerate(self.model.sources):
                stem_wave = sources[i]
                output_path = output_dir / f"{stem_name}.wav"
                torchaudio.save(str(output_path), stem_wave.cpu(), self.model.samplerate)
                print(f"[Demucs] Saved: {output_path}")
            
            # Erfolg
            status["status"] = "completed"
            status["done"] = True
            status["completed_at"] = datetime.now().isoformat()
            status["output_dir"] = str(output_dir)
            
            with open(status_path, 'w') as f:
                json.dump(status, f, indent=2)
            
            (output_dir / "DONE").touch()
            print(f"[Demucs] ✅ Fertig: {output_dir}")
            return status
            
        except Exception as e:
            status["status"] = "failed"
            status["error"] = str(e)
            status["failed_at"] = datetime.now().isoformat()
            with open(status_path, 'w') as f:
                json.dump(status, f, indent=2)
            print(f"[Demucs] ❌ Fehler: {e}")
            import traceback
            traceback.print_exc()
            raise

    def watch_and_process(self, input_base: Path, poll_interval: int = 10):
        input_base = Path(input_base)
        print(f"\n[Watcher] Starte Watcher")
        print(f"[Watcher] Input: {input_base}")
        print(f"[Watcher] Output: {self.output_base}")
        print(f"[Watcher] Modell: htdemucs auf {self.device}")
        print(f"[Watcher] Poll Interval: {poll_interval}s")
        
        processed = set()
        for done in self.output_base.glob("*/DONE"):
            processed.add(done.parent.name)
        print(f"[Watcher] Bereits verarbeitet: {processed}")
        
        try:
            while True:
                triggers = list(input_base.glob("_TRIGGER_*.json"))
                for trigger_path in triggers:
                    track_id = trigger_path.stem.replace("_TRIGGER_", "")
                    if track_id in processed:
                        continue
                    print(f"\n[Watcher] Neuer Trigger: {track_id}")
                    try:
                        with open(trigger_path, 'r') as f:
                            trigger_data = json.load(f)
                        print(f"[Watcher] Trigger Data: {trigger_data}")
                        
                        audio_candidates = list(input_base.glob(f"{track_id}_*"))
                        audio_candidates = [p for p in audio_candidates if not p.name.endswith(".json")]
                        if not audio_candidates:
                            print(f"[Watcher] Keine Audio Datei für {track_id}")
                            continue
                        audio_file = audio_candidates[0]
                        print(f"[Watcher] Audio: {audio_file}")
                        
                        result = self.separate_track(audio_file, track_id)
                        processed.add(track_id)
                        print(f"[Watcher] ✅ Erfolgreich: {track_id}")
                    except Exception as e:
                        print(f"[Watcher] Fehler bei {track_id}: {e}")
                        import traceback
                        traceback.print_exc()
                time.sleep(poll_interval)
        except KeyboardInterrupt:
            print("\n[Watcher] Beendet")

print("DemucsProcessor Klasse definiert")
check_gpu()


## 6. Watcher starten - Automatisierte Verarbeitung
Dieser Watcher läuft endlos und verarbeitet neue Uploads vom Desktop Editor automatisch

In [ ]:
from pathlib import Path

processor = DemucsProcessor(
    output_base=Path("/content/drive/MyDrive/AudioEditor_Stems/Output")
)

# Starte Watcher - läuft bis manuell gestoppt
processor.watch_and_process(
    input_base=Path("/content/drive/MyDrive/AudioEditor_Stems/Input"),
    poll_interval=10
)


## 7. Einzelnen Track manuell verarbeiten (Alternative zum Watcher)
Falls du einen spezifischen Track testen willst

In [ ]:
# Beispiel für manuelle Verarbeitung eines einzelnen Tracks
# from pathlib import Path
# processor = DemucsProcessor(output_base=Path("/content/drive/MyDrive/AudioEditor_Stems/Output"))
# result = processor.separate_track(
#     input_path=Path("/content/drive/MyDrive/AudioEditor_Stems/Input/<Track_ID>_song.mp3"),
#     track_id="<Track_ID>"
# )
# print(result)
print("Manuelle Verarbeitung - bitte Pfade anpassen und auskommentierten Code ausführen")


## 8. Output prüfen

In [ ]:
from pathlib import Path
import json

OUTPUT_BASE = Path("/content/drive/MyDrive/AudioEditor_Stems/Output")

for track_dir in OUTPUT_BASE.iterdir():
    if track_dir.is_dir():
        print(f"\n=== Track: {track_dir.name} ===")
        for f in track_dir.iterdir():
            print(f"  {f.name} - {f.stat().st_size / 1024/1024:.2f} MB")
        status_file = track_dir / "status.json"
        if status_file.exists():
            with open(status_file) as jf:
                print(f"  Status: {json.load(jf)}")
